# 02 — Question bank
One record per selected survey item. Preserve wording, context and option order. Metadata only: no human answers are loaded.


In [1]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
source_manifest = json.loads((ROOT / "configs/data_source.json").read_text())
split = json.loads((ROOT / "data/splits/question_split.json").read_text())
catalogue_file = next(f for f in source_manifest["files"] if f["repo_path"].endswith("question_catalog.json"))
catalogue_path = ROOT / catalogue_file["local_path"]
assert hashlib.sha256(catalogue_path.read_bytes()).hexdigest() == catalogue_file["sha256"]
assert source_manifest["revision"] == split["source_revision"]
catalogue = json.loads(catalogue_path.read_text())

training = set(split["training_columns"])
validation_policy = set(split["validation_policy"]["columns"])
validation_scale = set(split["validation_maximization"]["columns"])
test = set(split["final_test"]["columns"])
anchors = {c for group in split["anchor_groups"].values() for c in group}
selected = training | validation_policy | validation_scale | test
assert len(selected) == 441 and len(anchors) == 92


## 1. Source lookup
CSV column names identify items. A parent key includes block and type because some QIDs are reused.


In [2]:
def source_key(question):
    return "::".join([question["BlockName"].strip(), question["QuestionType"], question["QuestionID"]])

catalogue_by_key = {source_key(q): q for q in catalogue}
assert len(catalogue_by_key) == len(catalogue)

def find_source(qid):
    matches = [q for q in catalogue if q["QuestionID"] == qid]
    assert len(matches) == 1, f"Specify block/type for reused QID: {qid}"
    return matches[0]

def context(qid, stop_before=None):
    question = find_source(qid)
    text = question["QuestionText"]
    end = len(text) if stop_before is None else text.index(stop_before)
    return {"source_key": source_key(question), "start": 0, "end": end, "text": text[:end]}

selected_parents = [q for q in catalogue if selected.intersection(q.get("csv_columns", []))]
assert len(selected_parents) == 67


## 2. Carry forward instructions
Use exact source text. For receiver games, carry the rules and role description, not an earlier offer amount.


In [3]:
# Section introductions present in the catalogue, assigned to their original groups.
section_groups = {
    "QID260": ["QID25", "QID26", "QID29", "QID35", "QID232", "QID235"],
    "QID285": ["QID27", "QID30", "QID233"],
    "QID266": [*[f"QID{i}" for i in range(224, 231)], *[f"QID{i}" for i in range(149, 153)]],
    "QID242": ["QID84", *[f"QID{i}" for i in range(244, 249)],
               "QID250", "QID251", "QID252", *[f"QID{i}" for i in range(276, 280)],
               *[f"QID{i}" for i in range(117, 123)]],
}
instructions = {q["QuestionID"]: [] for q in selected_parents}
for intro, qids in section_groups.items():
    for qid in qids:
        instructions[qid].append(context(intro))

for qid in ["QID84", *[f"QID{i}" for i in range(244, 249)]]:
    instructions[qid].append(context("QID243"))
for qid in ["QID250", "QID251", "QID252", *[f"QID{i}" for i in range(276, 280)]]:
    instructions[qid].append(context("QID249"))

# Preserve the common game rules, excluding the sender's final question.
for qid in [f"QID{i}" for i in range(225, 231)]:
    instructions[qid].append(context("QID224", "In this scenario,"))
    if qid != "QID225":
        instructions[qid].append(context("QID225", "If the person offers"))
for qid in [f"QID{i}" for i in range(118, 123)]:
    instructions[qid].append(context("QID117", "In this scenario,"))
    if qid != "QID118":
        instructions[qid].append(context("QID118", "If the person sends"))


## 3. Expand matrices and resolve options
Keep one-based response codes and zero-based model indices. For bipolar rows, code 1 means the left choice and code 2 the right choice. No reverse-scoring or typo correction.


In [4]:
def role_for(column):
    if column in training:
        return "training", "wave1_3"
    if column in validation_policy:
        return "validation", "wave4"
    if column in validation_scale:
        return "validation", "wave1_3"
    assert column in test
    return "test", "wave4"

items = []
for parent in selected_parents:
    qid = parent["QuestionID"]
    matrix = parent["QuestionType"] == "Matrix"
    bipolar = matrix and parent["Settings"]["Selector"] == "Bipolar"
    columns = parent["csv_columns"]
    if matrix:
        assert len(parent["Rows"]) == len(columns)
        if "RowsID" in parent:
            assert columns == [f"{qid}_{row_id}" for row_id in parent["RowsID"]]
    else:
        assert parent["QuestionType"] == "MC" and len(columns) == 1

    for position, column in enumerate(columns):
        if column not in selected:
            continue
        row = parent["Rows"][position] if matrix else None
        source_labels = parent["Columns"] if matrix else parent["Options"]
        if bipolar:
            assert source_labels == ["1", "2"] and row.count(":") == 1
            labels = row.split(":", 1)
        else:
            labels = source_labels
        role, label_wave = role_for(column)
        items.append({
            "item_id": column,
            "parent_key": source_key(parent),
            "question_id": qid,
            "block": parent["BlockName"].strip(),
            "question_type": parent["QuestionType"],
            "response_format": "bipolar_choice" if bipolar else "rating_scale" if matrix else "single_choice",
            "role": role, "label_wave": label_wave, "is_anchor": column in anchors,
            "row_position": position + 1 if matrix else None,
            "row_id": column[len(qid) + 1:] if matrix else None,
            "stem": parent["QuestionText"],
            "row_text": row,
            "instructions": instructions[qid],
            # These stems refer explicitly to the whole list. Other matrices render one row.
            "matrix_context_rows": parent["Rows"] if bipolar or qid == "QID29" else [],
            "options": [
                {"code": index + 1, "index": index, "label": label,
                 "source_label": source_labels[index],
                 "side": ["left", "right"][index] if bipolar else None}
                for index, label in enumerate(labels)
            ],
        })

# Save original parent/context records once, unchanged, for provenance.
used_keys = {item["parent_key"] for item in items}
used_keys.update(c["source_key"] for item in items for c in item["instructions"])
sources = {key: catalogue_by_key[key] for key in sorted(used_keys)}
by_id = {item["item_id"]: item for item in items}
assert len(by_id) == len(items) == 441


## 4. Build readable question text
Use `question_text` for model input. `sources` retains the complete original records for audit, including all matrix rows.


In [5]:
def render_question(item):
    parts = []
    if item["instructions"]:
        parts.append("Instructions:\n" + "\n\n".join(c["text"] for c in item["instructions"]))
    parts.append("Question:\n" + item["stem"])
    if item["matrix_context_rows"]:
        parts.append("Full list (answer only the indicated row):\n" +
                     "\n".join(item["matrix_context_rows"]))
    if item["row_text"] is not None:
        parts.append("Indicated row:\n" + item["row_text"])
    parts.append("Answer options:\n" + "\n".join(
        f"{option['code']}. {option['label']}" for option in item["options"]
    ))
    return "\n\n".join(parts)

for item in items:
    item["question_text"] = render_question(item)

# Preview only; saved wording is never truncated.
preview_ids = ["QID84_4", "QID250_1", "QID279_1", "QID119", "QID287_10"]
display(pd.DataFrame([
    {"item": key, "role": by_id[key]["role"],
     "context_sources": [sources[c["source_key"]]["QuestionID"] for c in by_id[key]["instructions"]],
     "options": [o["label"] for o in by_id[key]["options"]]}
    for key in preview_ids
]))


,item,role,context_sources,options
0,QID84_4,training,"[QID242, QID243]","[$6.00 in 6 weeks, $3.00 in 5 weeks]"
1,QID250_1,training,"[QID242, QID249]","[Lottery, $0.50]"
2,QID279_1,training,"[QID242, QID249]","[Lottery with x=$7.00, $0]"
3,QID119,training,"[QID242, QID117, QID118]",[I would keep $0 for myself and send $12 to th...
4,QID287_10,validation,[],"[Strongly oppose, Somewhat oppose, Neither opp..."


## 5. Check wording, mappings and boundaries
Check source equality, non-consecutive row IDs, game context and exact held-out text overlap. These checks do not establish model accuracy or exhaust all semantic leakage risks.


In [6]:
assert set(by_id) == selected
assert Counter(item["role"] for item in items) == {"training": 430, "validation": 8, "test": 3}
assert {item["item_id"] for item in items if item["is_anchor"]} == anchors
assert not training.intersection(split["forbidden_input_columns_all_respondents_all_waves"])
assert all(sources[key] == catalogue_by_key[key] for key in sources)

for item in items:
    parent = sources[item["parent_key"]]
    assert item["stem"] == parent["QuestionText"]
    if item["row_position"] is not None:
        i = item["row_position"] - 1
        assert parent["csv_columns"][i] == item["item_id"]
        assert parent["Rows"][i] == item["row_text"]
    for c in item["instructions"]:
        assert c["text"] == sources[c["source_key"]]["QuestionText"][c["start"]:c["end"]]
    if item["matrix_context_rows"]:
        assert item["matrix_context_rows"] == parent["Rows"]
    options = item["options"]
    assert [o["code"] for o in options] == list(range(1, len(options) + 1))
    assert [o["index"] for o in options] == list(range(len(options)))
    assert all(isinstance(o["label"], str) and o["label"].strip() for o in options)
    original_labels = parent["Columns"] if parent["QuestionType"] == "Matrix" else parent["Options"]
    assert [o["source_label"] for o in options] == original_labels
    expected = item["row_text"].split(":", 1) if item["response_format"] == "bipolar_choice" else original_labels
    assert [o["label"] for o in options] == expected
    assert item["question_text"] == render_question(item)

assert by_id["QID84_4"]["row_position"] == 1
assert [o["label"] for o in by_id["QID84_4"]["options"]] == ["$6.00 in 6 weeks", "$3.00 in 5 weeks"]
assert by_id["QID287_10"]["row_position"] == 8
assert "2% tax" in by_id["QID287_10"]["row_text"]
assert "50% chance of LOSING $8" in by_id["QID279_1"]["question_text"]
assert by_id["QID279_1"]["options"][0]["label"] == "Lottery with x=$7.00"
assert by_id["QID233_1"]["options"][-1]["label"] == "Agree strongy"  # source typo preserved
assert "would then be tripled" in by_id["QID119"]["question_text"]
assert "If the person sends me $5" not in by_id["QID119"]["question_text"]
assert "both receive nothing" in by_id["QID226"]["question_text"]
assert "If the person offers to keep $0" not in by_id["QID226"]["question_text"]

training_text = "\n".join(item["question_text"].casefold() for item in items if item["role"] == "training")
assert all(item["row_text"].casefold() not in training_text for item in items if item["role"] != "training")
print("Passed: 441 items; source text, options, row IDs and context mappings verified.")


Passed: 441 items; source text, options, row IDs and context mappings verified.


## 6. Save
One question-bank JSON and a short audit report. Human responses and existing splits remain untouched.


In [7]:
bank = {
    "schema_version": 1,
    "source_repo": source_manifest["repo_id"],
    "source_revision": source_manifest["revision"],
    "catalogue_sha256": catalogue_file["sha256"],
    "question_split_sha256": hashlib.sha256((ROOT / "data/splits/question_split.json").read_bytes()).hexdigest(),
    "response_code_base": 1,
    "model_index_base": 0,
    "sources": sources,
    "items": items,
}
destination = ROOT / "data/processed/question_bank.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(bank, ensure_ascii=False, indent=2) + "\n")
assert json.loads(destination.read_text()) == bank

audit = {
    "status": "passed", "source_revision": source_manifest["revision"],
    "items": len(items), "parent_questions": len(selected_parents), "source_records": len(sources),
    "roles": dict(Counter(item["role"] for item in items)),
    "persona_evidence_items": sum(item["is_anchor"] for item in items),
    "option_count": sum(len(item["options"]) for item in items),
    "bipolar_items": sum(item["response_format"] == "bipolar_choice" for item in items),
    "items_with_carried_instructions": sum(bool(item["instructions"]) for item in items),
    "checks": ["source equality", "option order and codes", "matrix row mapping",
               "instruction source spans", "receiver-game context", "exact held-out text exclusion"],
    "human_response_files_loaded": False,
    "bank_path": str(destination.relative_to(ROOT)),
    "bank_sha256": hashlib.sha256(destination.read_bytes()).hexdigest(),
}
(ROOT / "outputs").mkdir(exist_ok=True)
(ROOT / "outputs/question_bank_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
display(pd.DataFrame(list(audit["roles"].items()), columns=["role", "items"]))
print(f"Saved {destination.relative_to(ROOT)}")


,role,items
0,training,430
1,validation,8
2,test,3


Saved data/processed/question_bank.json
